In [3]:
import pandas as pd
import os

# Load CSV
df = pd.read_csv("train.csv")

# Convert filenames FIRST
df["filename"] = df["filename"].str.replace(".ogg", ".wav", regex=False)

# Set audio directory
audio_dir = "/home/users/ss1482/sangcs372final/Finalproject/birdtrain_wav"

# Build full filepaths
df["filepath"] = df["filename"].apply(lambda x: os.path.join(audio_dir, x))

In [4]:
import torch
from torch.utils.data import Dataset
import numpy as np
import soundfile as sf
import librosa

TARGET_SR = 32000
CLIP_DURATION = 10
CLIP_LEN = TARGET_SR * CLIP_DURATION


class BirdDataset(Dataset):
    def __init__(self, df, label_to_idx):
        self.df = df.reset_index(drop=True)
        self.label_to_idx = label_to_idx

    def __len__(self):
        return len(self.df)

    def load_audio(self, filepath):
        try:
            audio, sr = sf.read(filepath)
        except:
            return np.zeros(CLIP_LEN, dtype=np.float32)

        # Stereo → mono
        if audio.ndim == 2:
            audio = np.mean(audio, axis=1)

        # Resample
        if sr != TARGET_SR:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)

        # float32
        audio = audio.astype(np.float32)

        # Pad / trim
        if len(audio) < CLIP_LEN:
            audio = np.pad(audio, (0, CLIP_LEN - len(audio)))
        else:
            audio = audio[:CLIP_LEN]

        return audio

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load audio
        audio = self.load_audio(row["filepath"])
        audio = torch.tensor(audio)

        # One-hot label
        label = torch.zeros(len(self.label_to_idx))
        label_idx = self.label_to_idx[row["primary_label"]]
        label[label_idx] = 1.0

        return audio, label

In [5]:
labels = sorted(df["primary_label"].unique())
label_to_idx = {label: i for i, label in enumerate(labels)}

In [6]:
dataset = BirdDataset(df, label_to_idx)


In [9]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=1
)

In [10]:
for audio, label in loader:
    print(audio.shape)  # (16, 320000)
    print(label.shape)  # (16, num_classes)
    break

torch.Size([16, 320000])
torch.Size([16, 206])
